# Алгоритм

In [2]:
import pandas as pd
import os
import openpyxl
from scipy.interpolate import interp1d

class SteelSortament:
    def __init__(self, root, files_of_type = ['.xlsx']):
        self.root = root
        self.files_of_type = files_of_type
    
    def all_profiles(self):
        res = []
        full_root_path = self.root
        for cwd, folders, files in os.walk(full_root_path):
            for fname in files:
                if os.path.splitext(fname)[1] in self.files_of_type:
                    res.append(os.path.join(cwd, fname))
        df = pd.concat([pd.read_excel(i) for i in res])
        df.drop_duplicates(inplace=True)
        
        # Явно приводим типы к float
        float_cols = ['A', 'P', 'Ix', 'Wx', 'Iy', 'Wy']
        for col in float_cols:
            if col in df.columns:
                df[col] = pd.to_numeric(df[col], errors='coerce')
        
        return df


def get_profile_data(df, profile):
    df = df.loc[df['profile'] == profile]
    if df.empty:
        return None

    res = (df['A'].values[0], df['Wx'].values[0], df['Ix'].values[0],df['s'].values[0],df['Wy'].values[0])
    
    return res




def get_Ry(s,grade):
    dict_ = {'235':([2,4],[230]),
             '245':([2,20],[240]),
             '255':([2,4,10,20,40],[250,240,240,230]),
             '345':([2,10,20,40,60,80,160],[340,320,300,280,270,260]),
             '345К':([4,10],[340]),
             '355':([8,16,40,60,80,100,160],[350,340,330,320,310,285]),
             '390':([8,50],[380]),
             '440':([8,50],[430]),
             '550':([8,50],[525])}

    try:
        Ry_list = dict_[(str(grade))][1]
        t_list =  dict_[(str(grade))][0]
    except KeyError:
        print(f'Сталь {grade} отсутствует в каталоге')
        return 1
    try:
        res = [Ry_list[i] for i in range(0, len(t_list)-1) if t_list[i] < s <= t_list[i+1]][0]
        return res
    except IndexError:
        #print(f'Изготовление листа толщиной {s} из стали {grade} не предусмотрено')
        return 1

def calculate_beam_ratios(span, q, w, grade, lim_deflection_ratio, safety_coefficient,
                          profile_name, df):
    
    profile_data = get_profile_data(df, profile_name)
    if profile_data is None:
        return None, None, None

    _, Wx, Ix, s, _ = profile_data
    Ry = get_Ry(float(s), grade)
    
    if Ry is None or pd.isna(Wx) or pd.isna(Ix): # Check if Ry or Wx/Ix from profile_data are None/NaN
        return None, None, None 

    M = (q * w) * span**2 / 8 * safety_coefficient
    M_ult = Ry * Wx / 10000
    M_ratio = M / M_ult

    E = 200000
    f = 5 / 384 * (q * w) * span**4 / (E * Ix) * 1000000
    f_ult = span * lim_deflection_ratio
    f_ratio = f / f_ult

    P = 0.1
    f_100_mm = (P * (span**3)) / (48 * E * Ix) *1000*1000*1000

    return M_ratio, f_ratio, f_100_mm


def apply_ratios_to_row(row, span, q, w, grade, lim_deflection_ratio, safety_coefficient, df):
    profile_name = row['profile']
    M_ratio, f_ratio, f_100_mm = calculate_beam_ratios(
        span, q, w, grade, lim_deflection_ratio, safety_coefficient, profile_name, df)
    
    return pd.Series({'M_ratio': M_ratio, 'f_ratio': f_ratio, 'f_100_mm':f_100_mm})


def filter_profiles( span, q, w, grade, lim_deflection_ratio, safety_coefficient, df,
                    Kmax=0.9, s_min=3.0, f_100=0.8, exclude=['Круглая труба','Двутавр дополнительный'] ):
    
    df_processed = df.copy()
    
    df_processed[['M_ratio', 'f_ratio', 'f_100_mm']] = df_processed.apply(
                apply_ratios_to_row,
                axis=1,
                args=(span, q, w, grade, lim_deflection_ratio, safety_coefficient, df) )
        
    filter_mask = (
                (df_processed['M_ratio'] <= Kmax) & (df_processed['M_ratio'] >= 0) &
                (df_processed['f_ratio'] <= Kmax) & (df_processed['f_ratio'] >= 0) &
                (~df_processed['type'].isin(exclude)) &
                (df_processed['s'] >= s_min) &
                (df_processed['f_100_mm'] <= f_100)
                )
    
    df_res = df_processed[filter_mask].copy()
    df_res.loc[:, 'steel'] = grade
    
    return df_res[['type','profile','M_ratio','f_ratio','f_100_mm','P','steel']]


def calculate_cost(span, q, w, grades, lim_deflection_ratio, safety_coefficient, df,
                   Kmax=0.9, s_min=3.0, f_100=0.8, exclude=['Круглая труба','Двутавр дополнительный']):
    
    cost = {'235': 67.500,
            '245': 67.500,
            '255': 70.000,
            '345': 82.500,
            '345К': 92.500,
            '355': 87.500,
            '390': 95.000,
            '440': 102.500,
            '550': 117.500}
    
    all_results = []
    for grade in grades:
        df_grade = filter_profiles(span, q, w, grade, lim_deflection_ratio,
                                   safety_coefficient, df, Kmax=Kmax, s_min=s_min, f_100=f_100, exclude=exclude)
        all_results.append(df_grade)
    
    res_df = pd.concat(all_results, ignore_index=True)
    res_df['cost'] = res_df.apply(lambda row: row['P']/1000 * cost[row['steel']], axis=1)
    res_df = res_df.sort_values(by='cost', ascending=True)
    
    return res_df

def allowable_deflection_ratio(l):
    span = [1, 3, 6, 12, 24]
    ratio = [120, 150, 200, 250, 300]
    f_linear = interp1d(span, ratio, kind='linear', bounds_error=False, fill_value=(120, 300))
    return float(f_linear(l))

sort = SteelSortament('Sortament/')
df = sort.all_profiles()
df

,profile,h,b,s,t,hw,bw,r,A,P,...,ix,Sx,Iy,Wy,Sy,iy,i0,X0,type,is_common
0,Тр. 10*1,10.0,10.0,1.0,1.0,NaN,NaN,NaN,0.282743,0.222,...,NaN,NaN,0.028981,0.057962,NaN,NaN,NaN,NaN,Круглая труба,1
1,Тр. 10*1.2,10.0,10.0,1.2,1.2,NaN,NaN,NaN,0.331752,0.260,...,NaN,NaN,0.032711,0.065421,NaN,NaN,NaN,NaN,Круглая труба,1
2,Тр. 10.2*1,10.2,10.2,1.0,1.0,NaN,NaN,NaN,0.289026,0.227,...,NaN,NaN,0.030940,0.060667,NaN,NaN,NaN,NaN,Круглая труба,1
3,Тр. 10.2*1.2,10.2,10.2,1.2,1.2,NaN,NaN,NaN,0.339292,0.266,...,NaN,NaN,0.034964,0.068557,NaN,NaN,NaN,NaN,Круглая труба,1
4,Тр. 12*0.8,12.0,12.0,0.8,0.8,NaN,NaN,NaN,0.281486,0.221,...,NaN,NaN,0.044362,0.073937,NaN,NaN,NaN,NaN,Круглая труба,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
178,180х110х12,180.0,110.0,12.0,12.0,NaN,NaN,NaN,33.690000,26.400,...,NaN,NaN,324.090000,38.200000,NaN,NaN,NaN,NaN,Уголок неравнополочный,1
179,200х125х11,200.0,125.0,11.0,11.0,NaN,NaN,NaN,34.870000,27.370,...,NaN,NaN,446.360000,45.980000,NaN,NaN,NaN,NaN,Уголок неравнополочный,1
180,200х125х12,200.0,125.0,12.0,12.0,NaN,NaN,NaN,37.890000,29.740,...,NaN,NaN,481.930000,49.850000,NaN,NaN,NaN,NaN,Уголок неравнополочный,1
181,200х125х14,200.0,125.0,14.0,14.0,NaN,NaN,NaN,1800.830000,34.430,...,NaN,NaN,57.430000,35.400000,NaN,NaN,NaN,NaN,Уголок неравнополочный,1


При определении fi_b значения за расчетную длину балки l_ef следует принимать расстояние между точками закреплений сжатого пояса от
поперечных смещений (узлами продольных или поперечных связей, точками крепления жесткого настила); при отсутствии связей l_ef=l
(где  - пролет балки); за расчетную длину консоли следует принимать:  при отсутствии закрепления сжатого пояса на конце
консоли в горизонтальной плоскости (здесь  - длина консоли) или расстояние между точками закрепления сжатого пояса в
горизонтальной плоскости - при закреплении пояса на конце и по длине консоли.

In [3]:
def calculate_lambda_b(l_ef, profile_name, df):
    profile_data = get_profile_data(df, profile_name)
    
    if profile_data is None:
        return None, None, None

    return profile_data
    
calculate_lambda_b(6, '30Б1', df)

(40.8, 424.0, 6318.22, 5.5, 59.33)

In [4]:
# условной гибкости сжатого пояса балки
lambda_b = l_ef/b (Ryf/E)**0.5

NameError: name 'l_ef' is not defined

In [5]:
#расчетное сопротивление стали элемента полки (пояса) растяжению, сжатию, изгибу по пределу текучести;
Ryf=get_Ryf(s,grade)


NameError: name 'get_Ryf' is not defined

In [6]:
# Коэффициент устойчивости при изгибе
fi_b = 1


# Подбор

In [5]:
span = 7.0 # Пролет балки, м
w = 3.0    # Шаг балок, м

# Общие параметры
grades = ['255','345','355']  # Список проверяемых марок сталей
lim_deflection_ratio = 1 / allowable_deflection_ratio(span) # Предельный прогиб

Kmax = 0.99 # максимальный коэффициент использвания
s_min = 2.0 # минимальная толщина стали
f_100 = 1.5 # Допустимый прогиб от 100 кг в центре пролета, мм

exclude = ['Круглая труба','Двутавр дополнительный','Двутавр с уклоном полок'] # Исключить типы

# Сбор нагрузок
loads = [['Сендвич-панел 120мм', 30, 1.3],
         ['Полезная', 70, 1.3]]

# Нагурзка нормативная, Т/м2
q = float(sum([i[1] for i in loads])) / 1000    # Нагурзка нормативная, Т/м2

# Коэффициент надежнгост
safety_coefficient = float (sum([i[1]*i[2] for i in loads]) / sum([i[1] for i in loads])) # коэффициент надежнгости
print (q,safety_coefficient)


calculate_cost(span, q, w, grades, lim_deflection_ratio, safety_coefficient, df, Kmax=Kmax, s_min=s_min, f_100=f_100, exclude=exclude).head(20)

0.1 1.3


,type,profile,M_ratio,f_ratio,f_100_mm,P,steel,cost
362,Швеллер,24П,0.409594,0.479612,1.227806,24.00,255,1.680000
173,Двутавр балочный,25Б1,0.348865,0.394579,1.010123,25.70,255,1.799000
63,Прямоугольная труба,240*120*5,0.462936,0.541167,1.385388,26.97,255,1.887900
363,Швеллер,27П,0.321069,0.333892,0.854765,27.70,255,1.939000
765,Швеллер,24П,0.289125,0.479612,1.227806,24.00,345,1.980000
64,Прямоугольная труба,240*120*5.5,0.425711,0.497565,1.273767,29.52,255,2.066400
57,Прямоугольная труба,220*140*5.5,0.435204,0.554939,1.420643,29.52,255,2.066400
174,Двутавр балочный,25Б2,0.307100,0.344463,0.881825,29.60,255,2.072000
556,Двутавр балочный,25Б1,0.246258,0.394579,1.010123,25.70,345,2.120250
210,Двутавр широкополочный,20Ш1,0.358930,0.518887,1.328350,30.60,255,2.142000


In [10]:
# Общие параметры
grades = ['255','345','355']  # Список проверяемых марок сталей
lim_deflection_ratio = 1/200 # Предельный прогиб
Kmax = 0.9 # максимальный коэффицинет использвания
s_min = 3.0 # минимальная толщина стали
f_100 = 0.8 # Допустимый прогиб от 100 кг в центре пролета, мм
exclude = ['Круглая труба','Двутавр дополнительный'] # Исключить типы

loads = [['Решетчатый настил', 28, 1.05],
         ['Полезная', 450, 1.2]]

# Нагурзка нормативная, Т/м2
q = float(sum([i[1] for i in loads])) / 1000    # Нагурзка нормативная, Т/м2

# Коэффициент надежнгост
safety_coefficient = float (sum([i[1]*i[2] for i in loads]) / sum([i[1] for i in loads])) # коэффициент надежнгости
print (q,safety_coefficient)

span = 1.35 # Пролет балки, м
w = 1.0    # Шаг балок, м

calculate_cost(span, q, w, grades, lim_deflection_ratio, safety_coefficient, df, Kmax=Kmax, s_min=s_min, f_100=f_100, exclude=exclude).head(20)

0.478 1.1912133891213388


,type,profile,M_ratio,f_ratio,f_100_mm,P,steel,cost
869,Уголок неравнополочный,80х50х5,0.701018,0.367754,0.615488,4.49,255,0.314300
865,Уголок неравнополочный,75х50х5,0.793664,0.439910,0.736251,4.79,255,0.335300
0,Квадратная труба,60*60*3,0.443475,0.436151,0.729960,5.19,255,0.363300
141,Прямоугольная труба,70*50*3,0.412453,0.347792,0.582078,5.19,255,0.363300
148,Прямоугольная труба,80*40*3,0.397294,0.293133,0.490599,5.19,255,0.363300
1771,Уголок неравнополочный,80х50х5,0.494836,0.367754,0.615488,4.49,345,0.370425
1767,Уголок неравнополочный,75х50х5,0.560233,0.439910,0.736251,4.79,345,0.395175
866,Уголок неравнополочный,75х50х6,0.668917,0.374225,0.626317,5.69,255,0.398300
789,Уголок равнополочный,75х75х5,0.749633,0.387384,0.648341,5.80,255,0.406000
766,Швеллер,6.5П,0.360323,0.313797,0.525183,5.90,255,0.413000
